# A First Look at the Housing Data

**DS4DH Practice Pack · Module 00 — Introduction to DH and Data Science**

*Technique:* Loading and inspecting a dataset with pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/00_first_look.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Before any statistic, any model, any claim: what is actually in this file?

This notebook is the one every other notebook in the pack assumes you have run.
It establishes the shape of the data, the vocabulary (STIR, CSD, CMA), and the
one structural trap that will otherwise corrupt every count you make.

The dataset is real: Canadian Census shelter-cost data for 194 geographies
across four Census Metropolitan Areas — Montréal, Toronto, Edmonton, Vancouver.

In [ ]:
# What columns are there, and what type is each?
print(df.dtypes.to_string())

## The vocabulary

| Term | Meaning |
|---|---|
| **STIR** | Shelter-cost-to-income ratio: the share of household income going to housing. 30% is the conventional affordability threshold. |
| **CSD** | Census Subdivision — a municipality. The fine-grained unit here. |
| **CMA** | Census Metropolitan Area — a city and its commuter zone. Contains many CSDs. |
| `Total` / `Owner` / `Renter` | STIR for all households, owners only, renters only. |
| `renter_owner_gap` | `Renter` − `Owner`, in percentage points. |
| `immigrant_status` | `Immigrant`, `Non-immigrants`, or `Total Immigrant Status` (everyone). |

Note that `Total` and `Total Immigrant Status` mean different things. `Total` is a
column (all tenures); `Total Immigrant Status` is a row category (all people).

In [ ]:
# How is the file organised?
print('immigrant_status values:')
print(df['immigrant_status'].value_counts().to_string())
print()
print('rows per CMA:')
print(df['cma'].value_counts().to_string())

## The trap

582 rows, but not 582 places.

Every geography appears **three times** — once per immigrant status. And some
rows are not CSDs at all: the four CMA totals and a Canada row are stacked in
the same table, identifiable only by an empty `csd_code`.

Count rows without handling both and you will report roughly three times as many
municipalities as exist, with national figures mixed into your city averages.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

## The pre-computed city summary

`city_summary.csv` holds the CMA-level averages. It is a convenience file — every
number in it can be rebuilt from `merged_dataset.csv`, and in Module 09 you will
see that rebuilding it a different way gives a different answer.

In [ ]:
print(df_city.to_string(index=False))

### 🔧 Your turn 1

Look at the `avg_renter_stir` column against `avg_owner_stir`.

In every city, renters spend a larger share of income on housing than owners.
Before running anything else: write down one sentence explaining what could
produce that gap other than "renting is worse value than owning".

## Missingness is not noise here

Statistics Canada suppresses cells for small populations, to prevent
re-identification. Those suppressions are not random — they concentrate in small
municipalities, which means the places most likely to vanish from your analysis
are systematically the small ones.

In [ ]:
cols = ['Total', 'Owner', 'Renter', 'tot_income', 'rent_income']
miss = pd.DataFrame({
    'missing': df[cols].isna().sum(),
    'pct': (df[cols].isna().mean() * 100).round(1),
})
print(miss.to_string())
print()
print('Renter STIR is missing more often than Owner STIR — small places often')
print('have too few renter households to report.')

### 🔧 Your turn 2

Change `cols` above to include `own_income` and `rent_pop`, and re-run.

Which column is missing most often? Does that change which questions this
dataset can answer?

In [ ]:
# Where does the missingness fall? Compare small and large places.
sized = df.dropna(subset=['csd_code', 'tot_pop'])
sized = sized[sized['immigrant_status'] == 'Total Immigrant Status']
small = sized[sized['tot_pop'] < sized['tot_pop'].median()]
large = sized[sized['tot_pop'] >= sized['tot_pop'].median()]

print(f'Renter STIR missing in smaller-than-median CSDs: {small["Renter"].isna().mean():.1%}')
print(f'Renter STIR missing in larger-than-median CSDs:  {large["Renter"].isna().mean():.1%}')

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The renter–owner gap is not a like-for-like comparison of two
ways to house the same household. Renters and owners differ in income, age,
household size, and location. Owners also have a mortgage that eventually ends,
while rent does not. Any of these can produce the gap without renting being
"worse value" — this is the confounding problem Module 05 addresses directly.

**Your turn 2.** `rent_income` is missing most often. That matters more than it
looks: it is the denominator for renter affordability, so the places where
renting is hardest to measure are exactly the small municipalities where housing
pressure is often most acute. The dataset can describe renter burden well in
large CSDs and poorly in small ones, and no amount of modelling fixes that.

The final cell shows the mechanism: suppression is roughly twice as common in
smaller-than-median CSDs. Your analysis sample is not a random sample of places.

</details>

## Where this stops

You now know the shape of the file and its two structural hazards: repeated
geographies and non-random missingness. You do not yet know anything about
housing. Every number above is a description of a spreadsheet, not a finding.

Next: **01a — Descriptive, Analytical and Policy Questions**, which is about
deciding what to ask before you have the means to answer it.